# Experimentos de detecção de veículos

Este notebook registra ground truth, benchmarks e visualizações. A aplicação final fica em `src/`.

A imagem usada é o JPEG extraído do PDF da prova (2048×1534 px). Não registre conclusões ou métricas sem executar as células correspondentes.

In [ ]:
%matplotlib widget

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import RectangleSelector
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

IMAGE_PATH = ROOT / "data/raw/drone_scene.jpg"
GROUND_TRUTH_PATH = ROOT / "data/annotations/ground_truth.json"
image = Image.open(IMAGE_PATH).convert("RGB")
image_width, image_height = image.size
print(f"Image: {image_width}x{image_height}")

## Anotação manual

Arraste para criar uma caixa de veículo. Use `d` para apagar a última caixa e `s` para salvar. Antes de salvar, revise toda a rua e o estacionamento; cada veículo deve aparecer exatamente uma vez.

In [ ]:
class ManualVehicleAnnotator:
    """Interactive rectangle annotator that persists vehicle-only ground truth."""

    def __init__(self, image: Image.Image, output_path: Path) -> None:
        self.image = image
        self.output_path = output_path
        self.boxes = self._load_existing()
        self.figure, self.axis = plt.subplots(figsize=(16, 12))
        self.axis.imshow(image)
        self.axis.set_title("Drag: add vehicle | d: delete last | s: save")
        self.axis.set_axis_off()
        self.selector = RectangleSelector(
            self.axis, self._on_select, useblit=True, button=[1], interactive=False
        )
        self.figure.canvas.mpl_connect("key_press_event", self._on_key)
        self._redraw()

    def _load_existing(self) -> list[list[float]]:
        if not self.output_path.exists():
            return []
        payload = json.loads(self.output_path.read_text(encoding="utf-8"))
        return [annotation["bbox_xyxy"] for annotation in payload["annotations"]]

    def _on_select(self, start, end) -> None:
        if None in (start.xdata, start.ydata, end.xdata, end.ydata):
            return
        x1, x2 = sorted((start.xdata, end.xdata))
        y1, y2 = sorted((start.ydata, end.ydata))
        if x2 - x1 >= 3 and y2 - y1 >= 3:
            self.boxes.append([round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)])
            self._redraw()

    def _on_key(self, event) -> None:
        if event.key == "d" and self.boxes:
            self.boxes.pop()
            self._redraw()
        elif event.key == "s":
            self.save()

    def _redraw(self) -> None:
        for patch in list(self.axis.patches):
            patch.remove()
        for index, (x1, y1, x2, y2) in enumerate(self.boxes, start=1):
            self.axis.add_patch(
                Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#00e5ff", lw=1.5)
            )
            self.axis.text(
                x1, y1, str(index), color="white", fontsize=8, bbox={"facecolor": "#004d5c"}
            )
        self.axis.set_title(f"{len(self.boxes)} vehicles | Drag: add | d: delete | s: save")
        self.figure.canvas.draw_idle()

    def save(self) -> None:
        sha256 = hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest()
        payload = {
            "schema_version": 1,
            "image": {
                "path": "data/raw/drone_scene.jpg",
                "width": image_width,
                "height": image_height,
                "sha256": sha256,
            },
            "annotations": [
                {"id": f"vehicle-{index:03d}", "category": "vehicle", "bbox_xyxy": box}
                for index, box in enumerate(self.boxes, start=1)
            ],
        }
        self.output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        print(f"Saved {len(self.boxes)} annotations to {self.output_path.relative_to(ROOT)}")


annotator = ManualVehicleAnnotator(image, GROUND_TRUTH_PATH)
plt.show()

## Próximas células

Após concluir a revisão manual, este notebook carregará os detectores, executará os benchmarks controlados e apresentará as métricas e comparações visuais.